# 08v2 - extends 08v1: mask-aware, adds medial_meniscus_tear, adds an ACL negative control

**Follows `08v1_meniscus_mcl_slot_group_check.ipynb`**, whose real
2026-08-28 Kaggle run gave two real findings (see its own "Real output"
section and [[project-rsna-phase-status]]):
1. Resolution (Nyquist, our own real `mm_per_px=0.5804`): resolves 2mm+
   features, not 1mm - a partial, not full, explanation.
2. Visual pattern across 7 real `mcl_injury` (`COR_T1`) rows and 6 real
   `lateral_meniscus_tear` (`SAG_FLUID_FS`) rows: **group 1 (centre)**
   repeatedly looked busier/brighter in a way consistent with the
   intercondylar-notch region (cruciate-ligament territory), while
   **groups 0/2 (outer)** more often showed a cleaner peripheral-
   compartment view with the meniscus wedge more clearly delineated.
   Plus one unplanned finding: **2/9 (22%) `mcl_injury`-positive gold
   studies have a completely blank `COR_T1` slot** (mask=0, not a bug -
   the cache legitimately doesn't have that slot for those studies).

That read came from eyeballing two single-observer images with no mask
filtering (some rows were blank and uninformative) and no
negative-control comparison. Before deciding whether to spend a Kaggle
retrain on this (user's explicit call, 2026-08-28: confirm more first),
this version:

1. **Filters by the cache's own presence mask** before choosing which
   studies to show, so every plotted row has real signal (blank studies
   are counted and reported separately, not wasted as blank rows).
2. **Adds `medial_meniscus_tear`** (same `SAG_FLUID_FS` slot as lateral
   meniscus, the third member of the weak cluster this notebook hasn't
   looked at yet).
3. **Adds an ACL negative control.** `acl_injury` is the healthiest
   finding in this comparison (pooled AUC 0.854) and is anatomically
   **central** (cruciate-ligament territory), unlike MCL/meniscus which
   are peripheral. Same `SAG_FLUID_FS` slot, same group 0/1/2 layout. If
   group 1 (centre) looks *more* complete/clean for ACL-positive studies
   than it did for meniscus-positive ones, that is real, non-anecdotal
   support for hypothesis 2, not just a story fitted to two images after
   the fact.

Still CPU-only, no training. Needs the cache Dataset attached to a
Kaggle kernel, same as `08v1`.

## Setup

Same convention as `08v1`/`04v2`. Also loads each shard's `_mask.npy`
this time (small files), to filter by slot presence before selecting
studies to visualize.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

_KAGGLE_RAW = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")
ON_KAGGLE = _KAGGLE_RAW.exists()
if not ON_KAGGLE:
    raise RuntimeError(
        "This notebook needs the published cache Dataset attached to a "
        "Kaggle kernel (multi-GB .npy files) - run on Kaggle, not locally."
    )

RAW_DIR = _KAGGLE_RAW
CACHE_DIR = Path("/kaggle/input/datasets/alherma7/cache-stevenleehans-rsna/cache")
if not (CACHE_DIR / "cache_meta.json").exists():
    raise RuntimeError(
        f"{CACHE_DIR} has no cache_meta.json - check the Dataset is "
        "attached to this kernel and CACHE_DIR still matches its mount path."
    )

with open(CACHE_DIR / "cache_meta.json") as f:
    meta = json.load(f)

SLOT_NAMES = meta["slots"]
CROP_MM = meta["crop_mm"]
IMG = meta["img"]
MM_PER_PX = CROP_MM / IMG
print("CACHE_DIR:", CACHE_DIR)
print("slots:", SLOT_NAMES)
print(f"crop_mm={CROP_MM}, img={IMG}, mm_per_px={MM_PER_PX:.4f}")

## Hypothesis 1 - resolution, against our own real numbers (unchanged from 08v1, kept for a self-contained record)

In [ ]:
for d_mm in [1.0, 2.0, 3.0, 5.0]:
    needed_mm_per_px = d_mm / 2
    resolves = MM_PER_PX <= needed_mm_per_px
    print(f"{d_mm:>4.1f} mm feature: needs <= {needed_mm_per_px:.3f} mm/px, "
          f"our cache has {MM_PER_PX:.3f} mm/px -> resolves: {resolves}")

## Select gold studies of interest

Four groups this time: `mcl_injury` (all 9 gold positives),
`medial_meniscus_tear` and `lateral_meniscus_tear` (sampled 6 each, seed
42 - the lateral sample is identical to 08v1's), and `acl_injury` as the
negative control (sampled 6, seed 42). Oversample each non-MCL group
before the mask filter below, so a study dropped for missing the slot
can be backfilled by the next one in the same seeded draw rather than
silently shrinking the figure.

In [ ]:
train_csv = pd.read_csv(RAW_DIR / "train.csv")
label_cols = [c for c in train_csv.columns if c not in ("StudyInstanceUID", "Report")]
gold_mask = train_csv[label_cols].notna().all(axis=1)
gold = train_csv.loc[gold_mask].reset_index(drop=True)
assert len(gold) == 58, len(gold)

mcl_pos_ids = gold.loc[gold["MCL"] == 1, "StudyInstanceUID"].tolist()
assert len(mcl_pos_ids) == 9, len(mcl_pos_ids)

rng = np.random.default_rng(42)


def oversampled_pool(column, n_target=6, over_factor=3):
    all_ids = gold.loc[gold[column] == 1, "StudyInstanceUID"].tolist()
    n_pool = min(len(all_ids), n_target * over_factor)
    return list(rng.choice(all_ids, size=n_pool, replace=False)), len(all_ids)


medial_men_pool, medial_men_total = oversampled_pool("Medial Meniscus")
lateral_men_pool, lateral_men_total = oversampled_pool("Lateral Meniscus")
acl_pool, acl_total = oversampled_pool("ACL")

print(f"gold MCL-positive: {len(mcl_pos_ids)} (using all)")
print(f"gold Medial Meniscus-positive: {medial_men_total} (pool of {len(medial_men_pool)} drawn)")
print(f"gold Lateral Meniscus-positive: {lateral_men_total} (pool of {len(lateral_men_pool)} drawn)")
print(f"gold ACL-positive (negative control): {acl_total} (pool of {len(acl_pool)} drawn)")

## Locate studies in the cache shards + mask-presence filter

Loads each shard's `_studies.csv` (row -> study id) and `_mask.npy`
(slot presence). Every study id used below is checked against the mask
for the slot it will be plotted in; absent ones are reported and
excluded from the figure instead of producing a blank row like in
`08v1`.

In [ ]:
shard_names = [f"train.s{i:02d}of04" for i in range(4)]
id_to_loc = {}
mask_by_shard = {}
for shard in shard_names:
    s = pd.read_csv(CACHE_DIR / f"{shard}_studies.csv")
    mask_by_shard[shard] = np.load(CACHE_DIR / f"{shard}_mask.npy")
    for row_idx, study_id in enumerate(s["StudyInstanceUID"]):
        id_to_loc[study_id] = (shard, row_idx)

for sid in mcl_pos_ids + medial_men_pool + lateral_men_pool + acl_pool:
    assert sid in id_to_loc, f"study missing from cache entirely: {sid}"
print("all candidate studies located in the cache - OK")


def slot_present(study_id, slot_name):
    shard, row_idx = id_to_loc[study_id]
    slot_idx = SLOT_NAMES.index(slot_name)
    return bool(mask_by_shard[shard][row_idx, slot_idx])


def filter_present(study_ids, slot_name, n_target=6):
    present = [sid for sid in study_ids if slot_present(sid, slot_name)]
    absent = [sid for sid in study_ids if sid not in present]
    return present[:n_target], absent


mcl_present, mcl_absent = filter_present(mcl_pos_ids, "COR_T1", n_target=9)
medial_men_present, medial_men_absent = filter_present(medial_men_pool, "SAG_FLUID_FS")
lateral_men_present, lateral_men_absent = filter_present(lateral_men_pool, "SAG_FLUID_FS")
acl_present, acl_absent = filter_present(acl_pool, "SAG_FLUID_FS")

for name, present, absent, slot in [
    ("mcl_injury", mcl_present, mcl_absent, "COR_T1"),
    ("medial_meniscus_tear", medial_men_present, medial_men_absent, "SAG_FLUID_FS"),
    ("lateral_meniscus_tear", lateral_men_present, lateral_men_absent, "SAG_FLUID_FS"),
    ("acl_injury (control)", acl_present, acl_absent, "SAG_FLUID_FS"),
]:
    print(f"{name}: {len(present)} with real {slot} signal, "
          f"{len(absent)} blank/absent in this draw -> {present if len(present) <= 9 else '(see above)'}")

In [ ]:
cache_by_shard = {shard: np.load(CACHE_DIR / f"{shard}_cache.npy", mmap_mode="r") for shard in shard_names}


def slot_slices(study_id, slot_name):
    """Return the (9, 224, 224) slice stack for one study's one slot."""
    shard, row_idx = id_to_loc[study_id]
    slot_idx = SLOT_NAMES.index(slot_name)
    return cache_by_shard[shard][row_idx, slot_idx]


import matplotlib.pyplot as plt


def plot_group_comparison(study_ids, slot_name, out_path, scale_bar_mm=3.0):
    n = len(study_ids)
    fig, axes = plt.subplots(n, 3, figsize=(9, 3 * n))
    if n == 1:
        axes = axes[None, :]
    group_labels = ["group 0 (low)", "group 1 (centre)", "group 2 (high)"]
    mid_channel = [1, 4, 7]  # middle slice of each 3-slice anchor group

    for row, study_id in enumerate(study_ids):
        slices = slot_slices(study_id, slot_name)  # (9, 224, 224)
        for col, mi in enumerate(mid_channel):
            axes[row, col].imshow(slices[mi], cmap="gray")
            axes[row, col].axis("off")
            if row == 0:
                axes[row, col].set_title(group_labels[col], fontsize=9)
        axes[row, 0].text(-14, IMG // 2, study_id[-8:], fontsize=7, rotation=90,
                           va="center", ha="center")

    bar_px = scale_bar_mm / MM_PER_PX
    x0, y0 = 10, IMG - 15
    axes[0, 0].plot([x0, x0 + bar_px], [y0, y0], color="red", linewidth=2)
    axes[0, 0].text(x0, y0 - 6, f"{scale_bar_mm:.0f} mm", color="red", fontsize=7)

    plt.suptitle(f"{slot_name} - centre vs. outer anchor groups", fontsize=11)
    plt.tight_layout()
    plt.savefig(out_path, dpi=100)
    plt.show()
    print(f"saved {out_path}")

## MCL (`COR_T1`) - mask-filtered

Should show up to 7 real rows (2 of the original 9 are blank per 08v1's
real run - no backfill possible, there are only 9 gold MCL positives
total).

In [ ]:
plot_group_comparison(mcl_present, "COR_T1", "mcl_group_comparison_v2.png")

## Medial meniscus (`SAG_FLUID_FS`) - new in this version

In [ ]:
plot_group_comparison(medial_men_present, "SAG_FLUID_FS", "medial_meniscus_group_comparison.png")

## Lateral meniscus (`SAG_FLUID_FS`) - mask-filtered re-run of 08v1's check

In [ ]:
plot_group_comparison(lateral_men_present, "SAG_FLUID_FS", "lateral_meniscus_group_comparison_v2.png")

## ACL (`SAG_FLUID_FS`) - negative control

**The key comparison.** Same slot as both meniscus checks above,
different (central, not peripheral) finding. **What to look for:** if
group 1 (centre) looks *cleaner/more complete* here than it did for
meniscus/MCL - not just different, but *specifically less busy/cluttered
in the way that looked like intercondylar-notch signal in 08v1* - that
is real support for hypothesis 2. If group 1 looks just as busy/messy
for ACL as it did for meniscus, the "centre = cruciate territory, good
for ACL" story does not hold up and hypothesis 2 is weaker than 08v1's
single-observer read suggested.

In [ ]:
plot_group_comparison(acl_present, "SAG_FLUID_FS", "acl_group_comparison_control.png")

## Real output (run 2026-08-28, on Kaggle, cache Dataset attached)

**Mask filtering worked as designed:** `mcl_group_comparison_v2.png` shows
the same 7 real (non-blank) rows `08v1` found by chance, now via the
mask filter rather than luck - no blank rows in any of the 4 panels
(`medial_meniscus_group_comparison.png`, `lateral_meniscus_group_comparison_v2.png`,
and `acl_group_comparison_control.png` all have 6/6 real rows).

**Visual read, all 4 panels (2 non-radiologist observers):**
- MCL (`COR_T1`, 7 rows) and both meniscus panels (`SAG_FLUID_FS`, 6
  rows each) reproduce `08v1`'s pattern: group 1 (centre) often looks
  busier/brighter in a way consistent with the intercondylar-notch
  region; groups 0/2 (outer) often show a cleaner peripheral-compartment
  view.
- **The `acl_injury` negative control (`SAG_FLUID_FS`, 6 rows) shows the
  same busy-centre appearance just as often** (clearly visible in rows
  98335840, 53419550, 48882465, 91102711 - all ACL-positive, a *central*
  structure). This is the key result of this notebook: the "busy centre"
  pattern is **not specific to peripheral findings** - it is most likely
  a generic feature of that middle sagittal position (near the
  intercondylar notch/fat pad) for any study, central or peripheral
  pathology alike.

**Conclusion: the negative control weakens hypothesis 2's clean
"centre=bad for peripheral, fine for central" story from `08v1`'s
single-pass read.** It does not disprove hypothesis 2 outright, but it
removes the visual "smoking gun." What still stands, unweakened by this
control: A3's cache samples all 3 anchor groups from a **narrow window**
(`RSNA_WINDOW=0.35,0.65`), so it remains plausible that **none of the 3
groups reach the true lateral/medial periphery** where MCL/lateral-
meniscus pathology commonly concentrates - reframed from "group 1 is
uniquely bad" to "the whole narrow window may be too central for these
structures, regardless of which of the 3 anchors within it is picked."

**Two non-radiologists have taken qualitative visual inspection as far
as it usefully goes here.** User's decision (2026-08-28): stop trying to
confirm further by eye, settle it empirically with a real training run.
Full design: `docs/superpowers/specs/2026-08-28-a2v2-multigroup-slot-attention-design.md`
(A2 v2 - all 3 groups on all 6 slots, 18 pseudo-slots, fold 0 pilot,
gated against A2 v1 fold 0's 0.7689).

Images not committed to the repo (real patient MRI data, same policy as
`sample_slot_cache_grid.png` and `08v1`'s images - see `.gitignore`).